In [1]:
import pandas as pd;
import matplotlib.pyplot as plt;
import os;
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
dataset = []
# Iterate by all csv files and merging them
for f in os.listdir("."):
  if f[-3:] == "csv":
    d1 = pd.read_csv(f)
    d1 = d1[(d1["BSSID"] == "b'IUE-WIFI") | (d1["BSSID"] == "bIUE-WIFI")].drop("BSSID", axis=1)
    d1["N"] = d1["N"].apply(lambda x : int(x[:-1]) if type(x) != int else x)
    dataset.append(d1.copy())
d1 = pd.concat(dataset)

In [4]:
# Check if all columns are right
d1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 457 entries, 1 to 473
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   DAY       457 non-null    object
 1   MONTH     457 non-null    int64 
 2   TIME      457 non-null    object
 3   SSID      457 non-null    object
 4   STRENGTH  457 non-null    int64 
 5   PLACE     457 non-null    int64 
 6   LOC       457 non-null    int64 
 7   N         457 non-null    int64 
dtypes: int64(5), object(3)
memory usage: 32.1+ KB


In [5]:
d1['PLACE'].value_counts()

,count
PLACE,
10102,135
10103,118
10101,103
10104,101


Entrenar: Un Random Forest, XGBoost, y NN

In [71]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

df_prep = d1.copy()

# Calculate 'Neighbouring' (number of APs detected in each specific measurement scan)
scan_counts = df_prep.groupby(['PLACE', 'LOC', 'N']).size().reset_index(name='Neighbouring')
df_prep = df_prep.merge(scan_counts, on=['PLACE', 'LOC', 'N'], how='left')

# Label encode SSID
le_ssid = LabelEncoder()
df_prep['SSID'] = le_ssid.fit_transform(df_prep['SSID'])

# Use SSID, STRENGTH, N, Neighbouring, and our spatial spot LOC as features
features = ['SSID', 'STRENGTH', 'N', 'Neighbouring', 'LOC']
X = df_prep[features]
y = df_prep['PLACE']

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

# Split into Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.2, random_state=42)
# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_val_scaled = scaler.transform(X_val)
print(f"Updated features with LOC: {features}")
print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")
print(f"Val set shape: {X_val_scaled.shape}")

Updated features with LOC: ['SSID', 'STRENGTH', 'N', 'Neighbouring', 'LOC']
Training set shape: (365, 5)
Test set shape: (73, 5)
Val set shape: (19, 5)


In [10]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

In [13]:
# Pivot the dataset so that each unique scan (PLACE, LOC, N) is a single row with SSIDs as features
pivoted_df = d1.pivot_table(
    index=['PLACE', 'LOC', 'N'],
    columns='SSID',
    values='STRENGTH',
    fill_value=-100
).reset_index()

# Features are the columns representing the SSIDs, plus LOC
feature_cols = [col for col in pivoted_df.columns if col not in ['PLACE', 'N']]
X_pivot = pivoted_df[feature_cols]
y_pivot = pivoted_df['PLACE']

# Label encode the target rooms
le_target_pivot = LabelEncoder()
y_pivot_encoded = le_target_pivot.fit_transform(y_pivot)

# Split into training and testing sets
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_pivot, y_pivot_encoded, test_size=0.2, random_state=42, stratify=y_pivot_encoded
)

# Scale the features
scaler_p = StandardScaler()
X_train_p_scaled = scaler_p.fit_transform(X_train_p)
X_test_p_scaled = scaler_p.transform(X_test_p)

print(f"Pivoted training fingerprint matrix shape: {X_train_p_scaled.shape}")
print(f"Pivoted testing fingerprint matrix shape:  {X_test_p_scaled.shape}")

Pivoted training fingerprint matrix shape: (64, 16)
Pivoted testing fingerprint matrix shape:  (16, 16)


In [16]:
# Train and test our models on the reconstructed WiFi fingerprints

# 1. Random Forest
rf_p = RandomForestClassifier(random_state=42)
rf_p.fit(X_train_p_scaled, y_train_p)
y_pred_rf_p = rf_p.predict(X_test_p_scaled)
print(f"Fingerprint - Random Forest Accuracy: {accuracy_score(y_test_p, y_pred_rf_p):.4f}")

# 2. XGBoost
xgb_p = XGBClassifier(eval_metric='mlogloss', random_state=42)
xgb_p.fit(X_train_p_scaled, y_train_p)
y_pred_xgb_p = xgb_p.predict(X_test_p_scaled)
print(f"Fingerprint - XGBoost Accuracy:       {accuracy_score(y_test_p, y_pred_xgb_p):.4f}")

# 3. Neural Network (MLP)
mlp_p = MLPClassifier(max_iter=1000, random_state=42)
mlp_p.fit(X_train_p_scaled, y_train_p)
y_pred_mlp_p = mlp_p.predict(X_test_p_scaled)
print(f"Fingerprint - Neural Network Accuracy: {accuracy_score(y_test_p, y_pred_mlp_p):.4f}")

Fingerprint - Random Forest Accuracy: 1.0000
Fingerprint - XGBoost Accuracy:       0.8750
Fingerprint - Neural Network Accuracy: 0.9375


In [120]:
from torch.utils.data import DataLoader, TensorDataset
ss = MinMaxScaler()
X_tensor = torch.FloatTensor(X_train_scaled)
y_tensor = torch.FloatTensor(y_train/3).reshape(-1, 1)
Xt_tensor = torch.FloatTensor(X_test_scaled)
yt_tensor = torch.FloatTensor(y_test/3).reshape(-1, 1)
Xv_tensor = torch.FloatTensor(X_val_scaled)
yv_tensor = torch.FloatTensor(y_val/3).reshape(-1, 1)
train_dataset = TensorDataset(X_tensor, y_tensor)
test_dataset = TensorDataset(Xt_tensor, yt_tensor)
val_dataset = TensorDataset(Xv_tensor, yv_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [132]:

class MLP(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size, dropout_rate=0.2):
        super(MLP, self).__init__()

        # Create list to hold all layers
        layers = []

        # Input layer
        layers.append(nn.Linear(input_size, hidden_sizes[0]))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout_rate))

        # Hidden layers
        for i in range(len(hidden_sizes)-1):
            layers.append(nn.Linear(hidden_sizes[i], hidden_sizes[i+1]))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))

        # Output layer
        layers.append(nn.Linear(hidden_sizes[-1], output_size))
        layers.append(nn.Softmax(dim=1))

        # Combine all layers into a sequential model
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

# Define model hyperparameters
input_size = 5  # Our data has 2 features
hidden_sizes = [32, 16, 8]
output_size = 4

# Create the model
model = MLP(input_size, hidden_sizes, output_size)
print(model)

MLP(
  (model): Sequential(
    (0): Linear(in_features=5, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=16, out_features=8, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=8, out_features=4, bias=True)
    (10): Softmax(dim=1)
  )
)


In [133]:
def train_model(model, train_loader, criterion, optimizer, num_epochs=100):
    # Lists to store metrics
    train_losses = []

    # Training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        running_loss = 0.0

        for inputs, targets in train_loader:
            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            # Squeeze targets to shape [batch_size] and cast to LongTensor for CrossEntropyLoss
            loss = criterion(outputs, targets.squeeze().long())

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        # Calculate epoch loss
        epoch_loss = running_loss / len(train_loader.dataset)
        train_losses.append(epoch_loss)

        # Print progress
        if (epoch+1) % 10 == 0:
            print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')

    return train_losses

# Configure model for CrossEntropyLoss (requires raw logits, so remove Softmax or use correct targets)
# Let's redefine criterion to CrossEntropyLoss
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Note: To get the best results, you should also remove the nn.Softmax(dim=1) layer from your MLP model class
# and recreate the model, as CrossEntropyLoss applies LogSoftmax internally.

# Prepare the data loader with integer targets (unscaled class labels 0 to 3)
X_tensor = torch.FloatTensor(X_train_scaled)
y_tensor = torch.LongTensor(y_train) # Keep original integer class labels [0, 1, 2, 3]
train_dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Train the model
train_losses = train_model(model, train_loader, criterion, optimizer, num_epochs=100)

Epoch 10/100, Loss: 1.3590
Epoch 20/100, Loss: 1.3010
Epoch 30/100, Loss: 1.2824
Epoch 40/100, Loss: 1.2748
Epoch 50/100, Loss: 1.2488
Epoch 60/100, Loss: 1.2491
Epoch 70/100, Loss: 1.2383
Epoch 80/100, Loss: 1.2241
Epoch 90/100, Loss: 1.2164
Epoch 100/100, Loss: 1.2063


In [134]:
def evaluate_model(model, test_loader, criterion):
    model.eval()  # Set model to evaluation mode
    test_loss = 0.0
    correct = 0
    total = 0

    # No gradient calculation needed for evaluation
    with torch.no_grad():
        for inputs, targets in test_loader:
            # Forward pass
            outputs = model(inputs)

            # Calculate loss
            loss = criterion(outputs, targets.squeeze().long())
            test_loss += loss.item() * inputs.size(0)

            # Calculate accuracy
            predicted = (outputs > 0.5).float()
            total += targets.size(0)
            correct += (predicted == targets).sum().item()

    # Calculate metrics
    avg_loss = test_loss / len(test_loader.dataset)
    accuracy = correct / total

    return avg_loss, accuracy

# Evaluate the model
test_loss, test_accuracy = evaluate_model(model, test_loader, criterion)
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')

Test Loss: 1.4784
Test Accuracy: 0.9589


In [107]:
# Pivot the dataset so that each unique scan (PLACE, LOC, N) is a single row with SSIDs as features
pivoted_df = d1.pivot_table(
    index=['PLACE', 'LOC', 'N'],
    columns='SSID',
    values='STRENGTH',
    fill_value=-100 # -100 represents a silent or undetected AP
).reset_index()

# Features are the columns representing the SSIDs, plus LOC
feature_cols = [col for col in pivoted_df.columns if col not in ['PLACE', 'N']]
X_pivot = pivoted_df[feature_cols]
y_pivot = pivoted_df['PLACE']

# Label encode the target rooms
le_target_pivot = LabelEncoder()
y_pivot_encoded = le_target_pivot.fit_transform(y_pivot)

# Split into training and testing sets
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X_pivot, y_pivot_encoded, test_size=0.2, random_state=42, stratify=y_pivot_encoded
)

# Scale the features
scaler_p = StandardScaler()
X_train_p_scaled = scaler_p.fit_transform(X_train_p)
X_test_p_scaled = scaler_p.transform(X_test_p)

print(f"Pivoted training fingerprint matrix shape: {X_train_p_scaled.shape}")
print(f"Pivoted testing fingerprint matrix shape:  {X_test_p_scaled.shape}")

Pivoted training fingerprint matrix shape: (64, 16)
Pivoted testing fingerprint matrix shape:  (16, 16)


In [130]:
# Train and test our models on the reconstructed WiFi fingerprints

# 1. Random Forest
rf_p = RandomForestClassifier(random_state=42, n_estimators=5)
rf_p.fit(X_train_p_scaled, y_train_p)
y_pred_rf_p = rf_p.predict(X_test_p_scaled)
print(f"Fingerprint - Random Forest Accuracy: {accuracy_score(y_test_p, y_pred_rf_p):.4f}")

# 2. XGBoost
xgb_p = XGBClassifier(eval_metric='mlogloss', random_state=42)
xgb_p.fit(X_train_p_scaled, y_train_p)
y_pred_xgb_p = xgb_p.predict(X_test_p_scaled)
print(f"Fingerprint - XGBoost Accuracy:       {accuracy_score(y_test_p, y_pred_xgb_p):.4f}")

# 3. Neural Network (MLP)
mlp_p = MLPClassifier(max_iter=1000, random_state=42, early_stopping=True, validation_fraction=0.3)
mlp_p.fit(X_train_p_scaled, y_train_p)
y_pred_mlp_p = mlp_p.predict(X_test_p_scaled)
print(f"Fingerprint - Neural Network Accuracy: {accuracy_score(y_test_p, y_pred_mlp_p):.4f}")

Fingerprint - Random Forest Accuracy: 0.8125
Fingerprint - XGBoost Accuracy:       0.8750
Fingerprint - Neural Network Accuracy: 0.8125
